# Closed loop v2 -- real Waymax scenarios, after the dropout + selection fixes

Runs the `meeting-prep` branch with the checkpoint from `Av2_retrain_meeting_prep.ipynb`
(`weights/stage1_weights_roi.pth`). Compared to the first closed-loop run (`Av2_closed_loop.ipynb`):

- **Tracker dropouts no longer disqualify scenarios.** Gaps <= 1 s are bridged (agent stays in the sim, path
  interpolated); longer gaps split the object into separate spans (it leaves and later re-enters). The first run
  could only use 2 of 60 scenarios, both straight.
- **The model sees the same ROI selection as in training** (nearest agents within 50 m, all map points within
  80 m). All agents are still *simulated*; only those within 50 m enter the MPC rollouts.
- **Two cost functions side by side**: `original` (unchanged) and `calibrated` (progress measured along the
  heading + a penalty on deviating from the reference's speed) -- aimed at the first run's ego that sped up to
  18 m/s while the human was nearly stopped. This is a first calibration, not a tuned one.

**Runtime**: MPC is pure numpy, ~1-3 s per step. Defaults: 3 turning + 3 straight scenarios x 2 costs x 40 steps
-> roughly 15-30 min. GPU only matters for the encoder forward pass.

## Setup

In [ ]:
from google.colab import auth
auth.authenticate_user()
!git config --global user.email "mayankarya510@gmail.com"
!git config --global user.name "meyenk"

In [ ]:
!pip install -q git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax

Clones the `meeting-prep` branch, not `main` -- the changes this notebook tests aren't merged yet.

In [ ]:
!git clone -b meeting-prep https://github.com/meyenk/waymax_experiment.git
%cd waymax_experiment
!git log --oneline -3

**(Optional) push access** -- only needed to push results/weights back from here. Skip if you'd rather download and commit yourself.

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/meyenk/waymax_experiment.git

## Sanity check (synthetic tests, no Waymax needed)
If any of these fail, stop -- the checkout is wrong or something regressed.

In [ ]:
!python -m tests.test_features
!python -m tests.test_transforms
!python -m tests.test_mpc
!python -m tests.test_closed_loop

## Load scenarios + the trained encoder

In [ ]:
import dataclasses, json, os, time
import numpy as np
import torch
import jax
from waymax import dataloader, config as _config

from src import features as F
from src.dataset import val_test_split
jax.config.update("jax_enable_x64", True)

# max_num_objects=128 = every object WOMD has. The old notebooks used 32, and
# Waymax keeps the first 32 in FILE order (not nearest, and with no guarantee
# the SDC is among them -- see the check below).
MAX_NUM_OBJECTS = 128
train_config = dataclasses.replace(
    _config.WOD_1_1_0_TRAINING,
    path="gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/training/training_tfexample.tfrecord@1000",
    max_num_objects=MAX_NUM_OBJECTS,
)
val_config = dataclasses.replace(
    _config.WOD_1_1_0_VALIDATION,
    path="gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord@150",
    max_num_objects=MAX_NUM_OBJECTS,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")
print(f"ROI: agents within {F.AGENT_RADIUS} m (max {F.MAX_AGENTS}), "
      f"map within {F.MAP_RADIUS} m (max {F.MAX_MAP_POINTS} points), lights max {F.MAX_LIGHTS}")

import pandas as pd
import matplotlib.pyplot as plt
from src.model import Stage1Model
from src.closed_loop import run_closed_loop, build_agent_paths, MAX_BRIDGE_GAP
from src.transforms import inverse_transform_positions
from src.visualize import is_turning_scenario
from src.train import get_ego_idx

SCENARIO_BUDGET = 300   # scan this many validation scenarios, keep the held-out test portion
test_scenarios = []
for i, scenario in enumerate(dataloader.simulator_state_generator(config=val_config)):
    if i >= SCENARIO_BUDGET:
        break
    if val_test_split(i) == "test":
        test_scenarios.append(scenario)
print(f"held-out test scenarios available: {len(test_scenarios)}")

CKPT = "weights/stage1_weights_roi.pth"
if not os.path.exists(CKPT):
    from google.colab import files
    print(f"{CKPT} not found -- upload it (from the retrain notebook)")
    os.makedirs("weights", exist_ok=True)
    up = files.upload()
    os.replace(next(iter(up)), CKPT)
cl_model = Stage1Model(hidden_dim=64, future_len=30).to(device)
cl_model.load_state_dict(torch.load(CKPT, map_location=device))
cl_model.eval()
print(f"loaded {CKPT}")

## Dropouts: what the first run hit, and what happens now
Same scan as before (objects whose validity has a hole inside their span), plus how the new handling splits them:
gaps <= 10 frames are bridged, longer ones become separate spans.

In [ ]:
n_with_gaps, total_gappy, total_objs, bridged_only, split = 0, 0, 0, 0, 0
gap_lengths = []
for s in test_scenarios:
    ei = get_ego_idx(s)
    valid = np.array(s.log_trajectory.valid)
    paths = build_agent_paths(s, ei)
    had_gap = False
    for i in range(valid.shape[0]):
        if i == ei or not valid[i].any():
            continue
        total_objs += 1
        runs = F.valid_runs(valid[i])
        if len(runs) > 1:
            had_gap = True
            total_gappy += 1
            gap_lengths += [b0 - a1 - 1 for (_, a1), (b0, _) in zip(runs[:-1], runs[1:])]
            if len(paths["spans"][i]) == 1:
                bridged_only += 1
            else:
                split += 1
    n_with_gaps += had_gap
g = np.array(gap_lengths)
print(f"scanned {len(test_scenarios)} scenarios ({total_objs} objects)")
print(f"  scenarios with >=1 object with a dropout: {n_with_gaps} ({100*n_with_gaps/len(test_scenarios):.0f}%)"
      " -- all of these were unusable before")
print(f"  objects with a dropout: {total_gappy} ({100*total_gappy/max(total_objs,1):.1f}%)")
print(f"    fully bridged (all gaps <= {MAX_BRIDGE_GAP} frames): {bridged_only}")
print(f"    split into spans (some gap longer): {split}")
if g.size:
    print(f"  gap length (frames) p50/p90/max: {np.percentile(g, 50):.0f} / {np.percentile(g, 90):.0f} / {g.max()}")

## Run the closed loop -- turning and straight, both cost functions

In [ ]:
T0 = 10          # first decision step (needs >= hist_len-1 = 9)
MAX_STEPS = 40   # 4 s of closed-loop driving at dt=0.1
N_CAND = 32
N_TURN, N_STRAIGHT = 3, 3
COSTS = {
    "original": None,
    "calibrated": {"progress_along_heading": True, "w_ref_speed": 1.0},
}

def pick_scenarios(scenarios, n_turn, n_straight, min_disp=10.0):
    turning, straight = [], []
    for s in scenarios:
        ei = get_ego_idx(s)
        if ei is None:
            continue
        traj = s.log_trajectory
        v = np.array(traj.valid[ei, T0:T0 + MAX_STEPS + 1])
        if not v.all():
            continue  # need the human's own path for comparison
        disp = np.hypot(traj.x[ei, T0 + MAX_STEPS] - traj.x[ei, T0], traj.y[ei, T0 + MAX_STEPS] - traj.y[ei, T0])
        if is_turning_scenario(s):
            turning.append(s)
        elif disp >= min_disp:  # skip parked/stopped egos for the straight set
            straight.append(s)
        if len(turning) >= n_turn and len(straight) >= n_straight:
            break
    return [("turn", s) for s in turning[:n_turn]] + [("straight", s) for s in straight[:n_straight]]

picked = pick_scenarios(test_scenarios, N_TURN, N_STRAIGHT)
print(f"picked {len(picked)} scenarios (no longer skipping ones with dropouts)\n")

runs = []
for j, (kind, s) in enumerate(picked):
    ei = get_ego_idx(s)
    for cost_name, ck in COSTS.items():
        np.random.seed(j)  # same candidate noise for both costs
        t_start = time.time()
        log = run_closed_loop(s, cl_model, ei, t0=T0, max_steps=MAX_STEPS, dt=0.1,
                              n_candidates=N_CAND, reactive=True, device=device, cost_kwargs=ck)
        label = f"{kind} #{j}"
        runs.append({"scenario": s, "log": log, "ego_idx": ei, "label": label, "cost": cost_name})
        print(f"  {label:12s} {cost_name:10s}: {len(log['ego_states'])-1} steps, "
              f"{len(log['agent_ids'][0])} agents simulated at t0, "
              f"{len(log['model_agent_ids'][0])} fed to the model, {time.time()-t_start:.0f}s")

## Summary
`max divergence` = how far the planner drifts from where the human went (some is expected: there's no route
conditioning, so it can't know the human's intent). `mean |v - v_human|` is the direct read on the speed
problem. `min dist to agent` below ~2 m is effectively a collision.

In [ ]:
def summarize(run, t0=T0):
    s, log, ei = run["scenario"], run["log"], run["ego_idx"]
    traj = s.log_trajectory
    ego = np.stack(log["ego_states"])
    n = ego.shape[0]
    logged = np.stack([np.array(traj.x)[ei, t0:t0 + n], np.array(traj.y)[ei, t0:t0 + n]], axis=-1)
    logged_v = np.hypot(np.array(traj.vel_x)[ei, t0:t0 + n], np.array(traj.vel_y)[ei, t0:t0 + n])
    div = np.linalg.norm(ego[:, :2] - logged, axis=-1)
    mins = [np.linalg.norm(a[:, :2] - ego[k, :2], axis=-1).min() if a.shape[0] else np.inf
            for k, a in enumerate(log["agent_states"])]
    modes = log["selection_modes"]
    return {
        "scenario": run["label"], "cost": run["cost"],
        "agents sim/model/mpc (t0)": f"{len(log['agent_ids'][0])}/{len(log['model_agent_ids'][0])}/{len(log['mpc_agent_ids'][0])}",
        "ego travelled (m)": round(float(np.sum(np.linalg.norm(np.diff(ego[:, :2], axis=0), axis=-1))), 1),
        "human travelled (m)": round(float(np.sum(np.linalg.norm(np.diff(logged, axis=0), axis=-1))), 1),
        "mean |v - v_human| (m/s)": round(float(np.mean(np.abs(ego[:, 3] - logged_v))), 2),
        "max divergence (m)": round(float(div.max()), 2),
        "final divergence (m)": round(float(div[-1]), 2),
        "min dist to agent (m)": round(float(np.min(mins)), 2),
        "blend/argmin": f"{modes.count('mppi_blend')}/{modes.count('argmin_fallback')}",
    }

summary = pd.DataFrame([summarize(r) for r in runs])
display(summary)
agg = summary.groupby("cost")[["mean |v - v_human| (m/s)", "max divergence (m)", "final divergence (m)",
                               "min dist to agent (m)"]].mean().round(2)
display(agg)

## Trajectories
Green = the human. Blue dashed = the planner. Thin salmon = the encoder's reference every 5th step -- separates
"planner ignored a good reference" from "the reference itself was wrong". Top row original cost, bottom calibrated.

In [ ]:
def plot_run(ax, run, t0=T0, ref_every=5):
    s, log, ei = run["scenario"], run["log"], run["ego_idx"]
    traj = s.log_trajectory
    ego = np.stack(log["ego_states"])
    n = ego.shape[0]
    rg = s.roadgraph_points
    rgv = np.array(rg.valid)
    ax.scatter(np.array(rg.x)[rgv], np.array(rg.y)[rgv], c="lightgray", s=2, zorder=1)
    logged = np.stack([np.array(traj.x)[ei, t0:t0 + n], np.array(traj.y)[ei, t0:t0 + n]], axis=-1)
    for k in range(0, len(log["ref_trajectories"]), ref_every):
        rw = inverse_transform_positions(log["ref_trajectories"][k], ego[k, :2], ego[k, 2])
        ax.plot(rw[:, 0], rw[:, 1], c="salmon", lw=0.9, alpha=0.75, zorder=2)
    last_ag = log["agent_states"][-1]
    if last_ag.shape[0]:
        ax.scatter(last_ag[:, 0], last_ag[:, 1], c="darkorange", s=22, zorder=3, label="agents (final, IDM)")
    ax.plot(logged[:, 0], logged[:, 1], c="green", lw=2.5, zorder=4, label="logged ego (human)")
    ax.plot(ego[:, 0], ego[:, 1], c="blue", lw=2.5, ls="--", zorder=5, label="closed-loop ego (MPC)")
    ax.scatter(ego[0, 0], ego[0, 1], c="black", marker="o", s=45, zorder=6, label="start")
    allxy = np.concatenate([logged, ego[:, :2]], axis=0)
    cx, cy = allxy[:, 0].mean(), allxy[:, 1].mean()
    span = max(np.ptp(allxy[:, 0]), np.ptp(allxy[:, 1])) / 2 + 25
    ax.set_xlim(cx - span, cx + span); ax.set_ylim(cy - span, cy + span)
    ax.set_aspect("equal")
    ax.set_title(f"{run['label']} -- {run['cost']}", fontsize=10)

cost_names = list(COSTS)
labels = list(dict.fromkeys(r["label"] for r in runs))
fig, axes = plt.subplots(len(cost_names), len(labels), figsize=(5 * len(labels), 5 * len(cost_names)), squeeze=False)
for r in runs:
    plot_run(axes[cost_names.index(r["cost"]), labels.index(r["label"])], r)
axes[0, 0].legend(loc="upper left", fontsize=7)
fig.suptitle("Closed-loop rollout vs logged human trajectory")
plt.tight_layout(); plt.show()

## Speed and divergence over time
Solid = planner, dotted = human. One column per cost function.

In [ ]:
fig, axes = plt.subplots(2, len(cost_names), figsize=(7 * len(cost_names), 8), squeeze=False)
for r in runs:
    col = cost_names.index(r["cost"])
    s, log, ei = r["scenario"], r["log"], r["ego_idx"]
    traj = s.log_trajectory
    ego = np.stack(log["ego_states"])
    n = ego.shape[0]
    steps = np.arange(n)
    logged_speed = np.hypot(np.array(traj.vel_x)[ei, T0:T0 + n], np.array(traj.vel_y)[ei, T0:T0 + n])
    logged = np.stack([np.array(traj.x)[ei, T0:T0 + n], np.array(traj.y)[ei, T0:T0 + n]], axis=-1)
    line, = axes[0, col].plot(steps, ego[:, 3], label=r["label"])
    axes[0, col].plot(steps, logged_speed, ls=":", color=line.get_color())
    axes[1, col].plot(steps, np.linalg.norm(ego[:, :2] - logged, axis=-1), label=r["label"])
for col, c in enumerate(cost_names):
    axes[0, col].set_title(f"speed ({c}): planner solid, human dotted"); axes[0, col].set_ylabel("m/s")
    axes[1, col].set_title(f"divergence from human path ({c})"); axes[1, col].set_ylabel("m")
    axes[1, col].set_xlabel("step")
    axes[0, col].legend(fontsize=7)
plt.tight_layout(); plt.show()

## Bird's-eye animation
Simulated ego + IDM agents vs the log, encoder waypoints, and -- new -- which agents the model was actually fed
that step (filled orange) vs simulated but outside its ROI (hollow orange). Change `WATCH` to pick a run.

In [ ]:
import mediapy

WATCH = 0   # index into `runs`

def render_closed_loop(run, t0=T0, window=55.0, dpi=95):
    s, log, ei = run["scenario"], run["log"], run["ego_idx"]
    traj = s.log_trajectory
    ego = np.stack(log["ego_states"])
    n = ego.shape[0]
    rg = s.roadgraph_points
    rgv = np.array(rg.valid)
    rg_x, rg_y = np.array(rg.x)[rgv], np.array(rg.y)[rgv]
    all_x, all_y = np.array(traj.x), np.array(traj.y)
    all_valid = np.array(traj.valid)

    frames = []
    fig, ax = plt.subplots(figsize=(6, 6), dpi=dpi)
    for k in range(n):
        t = t0 + k
        ex, ey, eyaw, ev = ego[k]
        ax.clear()
        ax.scatter(rg_x, rg_y, c="lightgray", s=2, zorder=1)
        vn = all_valid[:, t]
        ax.scatter(all_x[vn, t], all_y[vn, t], facecolors="none", edgecolors="darkgray",
                   s=55, linewidths=1.0, zorder=2, label="agents (logged)")
        ag, ids = log["agent_states"][k], log["agent_ids"][k]
        if ag.shape[0]:
            fed = np.isin(ids, log["model_agent_ids"][k]) if k < len(log["model_agent_ids"]) else np.zeros(len(ids), bool)
            ax.scatter(ag[fed, 0], ag[fed, 1], c="darkorange", s=30, zorder=3, label="simulated, fed to model")
            ax.scatter(ag[~fed, 0], ag[~fed, 1], facecolors="none", edgecolors="darkorange", s=30, zorder=3,
                       label="simulated, outside ROI")
        ax.plot(all_x[ei, t0:t + 1], all_y[ei, t0:t + 1], c="green", lw=1.6, zorder=4)
        ax.scatter(all_x[ei, t], all_y[ei, t], c="green", marker="*", s=170, zorder=5, label="ego (logged)")
        ax.plot(ego[:k + 1, 0], ego[:k + 1, 1], c="blue", lw=1.9, zorder=5)
        ax.scatter(ex, ey, c="blue", marker="*", s=220, zorder=7, label="ego (closed-loop)")
        ax.add_patch(plt.Circle((ex, ey), F.AGENT_RADIUS, fill=False, ls="--", lw=0.8, color="k"))
        if k < len(log["ref_trajectories"]):
            rw = inverse_transform_positions(log["ref_trajectories"][k], np.array([ex, ey]), eyaw)
            ax.plot(rw[:, 0], rw[:, 1], c="red", ls="--", lw=1.6, zorder=6, label="encoder waypoints")
        fut_end = min(t + 31, all_x.shape[1])
        ax.plot(all_x[ei, t:fut_end], all_y[ei, t:fut_end], c="limegreen", lw=1.6, zorder=6,
                label="human's actual next 3s")
        mode = log["selection_modes"][k] if k < len(log["selection_modes"]) else "-"
        ax.set_xlim(ex - window, ex + window); ax.set_ylim(ey - window, ey + window)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{run['label']} ({run['cost']})  t={t}  v={ev:.1f} m/s  {mode}", fontsize=9)
        ax.legend(loc="upper left", fontsize=6, framealpha=0.9)
        fig.canvas.draw()
        frames.append(np.asarray(fig.canvas.buffer_rgba())[..., :3].copy())
    plt.close(fig)
    return frames

frames = render_closed_loop(runs[WATCH])
print(f"rendered {len(frames)} frames for '{runs[WATCH]['label']}' ({runs[WATCH]['cost']})")
mediapy.show_video(frames, fps=10)

## Save summary

In [ ]:
os.makedirs("results", exist_ok=True)
summary.to_csv("results/closed_loop_meeting_prep.csv", index=False)
print(agg.to_markdown())